<a href="https://colab.research.google.com/github/Tayyaba614/-My-Portfolio-/blob/main/PSX_Banking_OHLCV_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install psxdata pandas numpy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Pandas installed!")
print("NumPy installed!")
print("Matplotlib installed!")

In [ ]:

import psxdata

print("PSXData imported successfully!")

In [ ]:

import psxdata

print(dir(psxdata))

In [ ]:

import psxdata
import inspect

print("Stocks module:", psxdata.stocks)
print("Available functions:")

print(dir(psxdata.stocks))

In [ ]:

import inspect
import psxdata

print("Function signature:")
print(inspect.signature(psxdata.stocks))

print("\nFunction documentation:")
print(inspect.getdoc(psxdata.stocks))

In [ ]:

import psxdata
import pandas as pd

# Download UBL historical data
ubl_data = psxdata.stocks(
    "UBL",
    start="2025-11-24",
    end="2026-11-24"
)

# Display results
print("UBL Historical Stock Data")
print(ubl_data.head())

print("\nDataset shape:")
print(ubl_data.shape)

print("\nColumn names:")
print(ubl_data.columns.tolist())

print("\nLast 5 records:")
print(ubl_data.tail())

In [ ]:

# Sort dates from oldest to newest
ubl_data["date"] = pd.to_datetime(ubl_data["date"])

ubl_data = ubl_data.sort_values(
    by="date"
).reset_index(drop=True)

# Check flagged records
print("Anomaly records:")
print(ubl_data[ubl_data["is_anomaly"] == True])

# Check chronological order
print("\nFirst 5 records:")
print(ubl_data.head())

print("\nLast 5 records:")
print(ubl_data.tail())

# Check missing values
print("\nMissing values:")
print(ubl_data.isnull().sum())

In [ ]:

import psxdata
import pandas as pd

companies = [
    "UBL", "MEBL", "HBL", "MCB",
    "NBP", "BOP", "BAHL", "BAFL"
]

start_date = "2025-11-24"
end_date = "2026-11-24"

all_data = []
failed_companies = []

for company in companies:
    try:
        print(f"Downloading {company}...")

        df = psxdata.stocks(
            company,
            start=start_date,
            end=end_date
        )

        if df.empty:
            print(f"No data available for {company}")
            failed_companies.append(company)
            continue

        df["date"] = pd.to_datetime(df["date"])
        df["Company"] = company

        df = df.sort_values("date")

        all_data.append(df)

        print(f"{company}: {len(df)} records downloaded")

    except Exception as e:
        print(f"Error downloading {company}: {e}")
        failed_companies.append(company)

# Combine all available companies
if all_data:
    banking_data = pd.concat(
        all_data,
        ignore_index=True
    )

    banking_data = banking_data.sort_values(
        ["date", "Company"]
    ).reset_index(drop=True)

    # Save combined dataset
    banking_data.to_csv(
        "PSX_Banking_OHLCV.csv",
        index=False
    )

    print("\nDATA COLLECTION COMPLETE")
    print("Total records:", len(banking_data))
    print("Companies collected:",
          banking_data["Company"].nunique())

    print("\nCompany-wise records:")
    print(banking_data.groupby("Company").size())

    print("\nMissing values:")
    print(banking_data.isnull().sum())

    print("\nFailed companies:", failed_companies)

    print("\nFirst 10 rows:")
    print(banking_data.head(10))

else:
    print("No company data was collected.")

In [ ]:

import psxdata

mcb_data = psxdata.stocks(
    "MCB",
    start="2025-11-24",
    end="2026-11-24",
    cache=False
)

print("MCB data shape:", mcb_data.shape)
print(mcb_data.head())

In [ ]:

# Check data quality

print("Total records:", len(banking_data))

print("\nCompanies:")
print(banking_data["Company"].unique())

# Check flagged anomalies
print("\nFlagged anomalies:")
print(
    banking_data[
        banking_data["is_anomaly"] == True
    ].groupby("Company").size()
)

# Check OHLC consistency
invalid_ohlc = banking_data[
    (banking_data["high"] < banking_data["open"]) |
    (banking_data["high"] < banking_data["close"]) |
    (banking_data["low"] > banking_data["open"]) |
    (banking_data["low"] > banking_data["close"]) |
    (banking_data["high"] < banking_data["low"])
]

print("\nInvalid OHLC records:", len(invalid_ohlc))
print(invalid_ohlc)

# Check duplicate dates
print("\nDuplicate company-date records:",
      banking_data.duplicated(
          subset=["Company", "date"]
      ).sum())

In [ ]:
# Add MCB to existing dataset
mcb_data["date"] = pd.to_datetime(mcb_data["date"])
mcb_data["Company"] = "MCB"

banking_data = pd.concat(
    [banking_data, mcb_data],
    ignore_index=True
)

banking_data = banking_data.sort_values(
    ["date", "Company"]
).reset_index(drop=True)

# Verify
print("Total records:", len(banking_data))
print("Companies:", banking_data["Company"].unique())
print("MCB records:",
      len(banking_data[banking_data["Company"] == "MCB"]))